In [56]:
import pandas as pd
import numpy as np
import sys, os
from importlib import reload
import yaml
from pathlib import Path
from dataclasses import asdict

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
from analysis import plots
import analysis.report
import training.gbm_model_trainer
from training.gbm_model_trainer import GBMModelTrainerConfig 

reload(analysis)
reload(analysis.plots)
reload(analysis.report)
reload(training.gbm_model_trainer)

<module 'training.gbm_model_trainer' from 'c:\\Users\\ASacco\\OneDrive - Plymouth Rock Assurance Corp\\repos\\analysis-tools\\src\\training\\gbm_model_trainer.py'>

## Load data

In [57]:
df = pd.read_csv("../data/bank-full.csv", delimiter=';')

In [58]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [59]:
# Convert 'y' to numeric for analysis
df["target"] = np.where(df["y"] == "no", 0, 1)

# Add weights to test weight parameter across functions 
df["weights"] = np.abs(np.random.randn(len(df.index)))

# Add an arbitrary data split for testing functions (TVH = 40/30/30)
df["random"] = np.random.uniform(0, 1, len(df.index))
df["split"] = np.where(
    df["random"]  > 0.70, 
    "V",
    np.where(
        df["random"]  > 0.40, 
        "H", 
        "T"
    )
)

In [60]:
df["split"].value_counts(dropna=False, normalize=True)

split
T    0.398885
H    0.302714
V    0.298401
Name: proportion, dtype: float64

In [61]:
df.groupby("split")["target"].mean()

split
H    0.115154
T    0.119497
V    0.115484
Name: target, dtype: float64

In [62]:
# Split data; define features and target
train = df.query("split == 'T'").drop(columns=["y", "split"])
test = df.query("split == 'V'").drop(columns=["y", "split"])
holdout = df.query("split == 'H'").drop(columns=["y", "split"])

## Use ModelTrainer class for training
- Can use example config files in `analysis-tools/examples` or create a config within your notebook and save it locally

In [183]:
# Define your config
from training.gbm_model_trainer import GBMModelTrainerConfig
config = GBMModelTrainerConfig(

    # Training parameters
    actual_col="target",
    predicted_col="pred_xgb",

    hyperparameters={
        "objective": "binary:logistic",
        "n_estimators": 100,
        "max_depth": 3,
        "learning_rate": 0.1,
        "subsample": 0.5,
        "colsample_bytree": 0.5,
        "random_state": 42,
    },
    feval="logloss",

    # Logging parameters
    output_log=True,
    output_report=True,

    # Reporting parameters
    report_params={
    },
    # tabulate_vars=["job", "marital", "education"],
    # plots_to_add=[
    #     {
    #         "plot": "plot_error_by_group_grid",
    #         "title": "Error by Analysis Variables",
    #         "kwargs": {
    #             "group_cols": ["job", "education", "age"]
    #         }
    #     },
    #     {
    #         "plot": "gain_curve_with_gini",
    #         "title": "Gain Curve / Lorenz Curve"
    #     },
    #     {
    #         "plot": "partial_gini_plot",
    #         "title": "Partial Gini (Top 15%)",
    #         "kwargs": {
    #             "top_percent": 15
    #         }
    #     },
    #     {
    #         "plot": "lift_chart",
    #         "title": "Lift Chart"
    #     },
    #     {
    #         "plot": "crunched_residual_plot",
    #         "title": "Crunched Residuals"
    #     },
    #     {
    #         "plot": "plot_residual_fit",
    #         "title": "Std and Avg of Normalized Residuals",
    #         "kwargs": {
    #             "residual_type": "normalized"
    #         }
    #     }
    # ],
)

# Output config to YAML
config_output_path = Path("../examples/slim_training_config.yaml")

import yaml
# Write config to YAML file
with config_output_path.open("w") as f:
    yaml.dump(asdict(config), f, indent=2)

print(f"Config saved to {config_output_path.resolve()}")

Config saved to C:\Users\ASacco\OneDrive - Plymouth Rock Assurance Corp\repos\analysis-tools\examples\slim_training_config.yaml


In [184]:
reload(training.gbm_model_trainer)
reload(analysis.report)
reload(analysis.plots)
from scoring import callbacks
reload(callbacks)

<module 'scoring.callbacks' from 'c:\\Users\\ASacco\\OneDrive - Plymouth Rock Assurance Corp\\repos\\analysis-tools\\src\\scoring\\callbacks.py'>

In [185]:
from training.gbm_model_trainer import GBMModelTrainer
from scoring import callbacks

mt_xgboost = GBMModelTrainer(
    config_path="../examples/slim_training_config.yaml",
    train_df=train,
    valid_df=test,
    holdout_df=holdout
)

In [186]:
mt_xgboost.config

GBMModelTrainerConfig(actual_col='target', predicted_col='pred_xgb', output_dir='outputs', log_file='training.log', report_file='model_analysis.html', model_file='model_obj.json', hyperparameters={'colsample_bytree': 0.5, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'objective': 'binary:logistic', 'random_state': 42, 'subsample': 0.5}, feval='logloss', output_log=True, base_margin_col=None, log_eval_period=50, early_stopping_rounds=None, early_stopping_metric=None, early_stopping_maximize=False, output_report=True, report_params={}, plots_to_add=[], tabulate_vars=[], tuning=TuningConfig(search_space={}, n_iter=10, objective_func=None, output_tuning=False, tuning_file='tuning.csv'))

In [187]:
mt_xgboost.train()

2025-07-31 13:02:30,470 [INFO] [0] validation-logloss: 0.33364
2025-07-31 13:02:30,546 [INFO] [50] validation-logloss: 0.21860
2025-07-31 13:02:30,600 [INFO] Training completed in 0.2121 seconds
2025-07-31 13:02:30,602 [INFO] Model: XGBoost Booster
2025-07-31 13:02:30,603 [INFO] Hyperparameters: {'colsample_bytree': 0.5, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'objective': 'binary:logistic', 'random_state': 42, 'subsample': 0.5}
2025-07-31 13:02:30,604 [INFO] Training data shape: (18034, 19)
2025-07-31 13:02:30,604 [INFO] Validation data shape: (13491, 19)
2025-07-31 13:02:30,605 [INFO] Number of predictors: 18
2025-07-31 13:02:30,606 [INFO] Target column: 'target'
2025-07-31 13:02:30,607 [INFO] Prediction column: 'pred_xgb'
2025-07-31 13:02:30,617 [INFO] Model saved to C:\Users\ASacco\OneDrive - Plymouth Rock Assurance Corp\repos\analysis-tools\notebooks\outputs\model_obj.json
2025-07-31 13:02:31,190 [INFO] Added plot: Gain Curve / Lorenz Curve (gain_curve_with_gini

Analysis report generated at outputs\model_analysis.html


In [188]:
mt_xgboost.log_lines

['2025-07-31 13:02:30,470 [INFO] [0] validation-logloss: 0.33364',
 '2025-07-31 13:02:30,546 [INFO] [50] validation-logloss: 0.21860',
 '2025-07-31 13:02:30,600 [INFO] Training completed in 0.2121 seconds',
 '2025-07-31 13:02:30,602 [INFO] Model: XGBoost Booster',
 "2025-07-31 13:02:30,603 [INFO] Hyperparameters: {'colsample_bytree': 0.5, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'objective': 'binary:logistic', 'random_state': 42, 'subsample': 0.5}",
 '2025-07-31 13:02:30,604 [INFO] Training data shape: (18034, 19)',
 '2025-07-31 13:02:30,604 [INFO] Validation data shape: (13491, 19)',
 '2025-07-31 13:02:30,605 [INFO] Number of predictors: 18',
 "2025-07-31 13:02:30,606 [INFO] Target column: 'target'",
 "2025-07-31 13:02:30,607 [INFO] Prediction column: 'pred_xgb'",
 '2025-07-31 13:02:30,617 [INFO] Model saved to C:\\Users\\ASacco\\OneDrive - Plymouth Rock Assurance Corp\\repos\\analysis-tools\\notebooks\\outputs\\model_obj.json',
 '2025-07-31 13:02:31,190 [INFO] Added

In [175]:
mt_xgboost.logger.handlers

[<StreamHandler stderr (NOTSET)>,
 <FileHandler c:\Users\ASacco\OneDrive - Plymouth Rock Assurance Corp\repos\analysis-tools\notebooks\outputs\training.log (NOTSET)>,
 <ListLogHandler (NOTSET)>]

## Test Hyperparameter tuning